# JSON Schema Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/json_schema.py`.

# `dereference_refs`

Resolves and inlines JSON Schema `$ref` objects without modifying the input schema.

```python
dereference_refs(
    schema_obj: dict[str, Any], # Schema object or schema fragment to process
    *,
    full_schema: dict[str, Any] | None = None, # Complete schema used to resolve references
    skip_keys: Sequence[str] | None = None, # Keys whose nested values should be copied without recursive processing
) -> dict[str, Any] # New schema dictionary with references resolved
```

When `full_schema` is omitted or falsy, `schema_obj` is used as the reference source.

When `skip_keys` is `None`, `"$defs"` is skipped during recursive processing and is preserved as a deep copy. Supplying a sequence skips exactly those keys; supplying an empty sequence recursively processes all keys.

Pure `$ref` objects are replaced by deep copies of their referenced values. For objects containing both `$ref` and additional properties, the referenced dictionary is resolved first and the additional properties are merged afterward, so the additional properties take precedence.

Circular references are broken when the same reference is encountered while it is already being resolved. At that point, the repeated `$ref` is removed and only its additional properties are retained.

Raises `ValueError` when a `$ref` path does not begin with `"#"`. Raises `KeyError` when a referenced path cannot be found.

In [ ]:
# 1. Resolve normal $ref values

from langchain_core.utils.json_schema import dereference_refs # Import the JSON Schema dereferencing utility


schema = { # Define a JSON Schema containing reusable definitions
    "type": "object", # Specify that the root value is an object
    "properties": { # Define the object's properties
        "name": {"type": "string"}, # Define a normal string property
        "address": {"$ref": "#/$defs/Address"}, # Reference the reusable Address schema
    }, # Finish defining the properties
    "$defs": { # Define reusable schema components
        "Address": { # Define the reusable Address schema
            "type": "object", # Specify that an address is an object
            "properties": { # Define address properties
                "city": {"type": "string"}, # Define the city property
                "pincode": {"type": "integer"}, # Define the pincode property
            }, # Finish defining address properties
        }, # Finish defining the Address schema
    }, # Finish defining reusable components
} # Finish creating the schema

resolved_schema = dereference_refs(schema) # Replace the address reference with its complete schema

resolved_schema # Display the dereferenced schema in Jupyter

In [ ]:
# 2. Override fields beside $ref
# Properties written beside $ref take precedence over the referenced schema.

schema_with_override = { # Define a schema containing a reference with extra properties
    "$defs": { # Define reusable schema components
        "UserName": { # Define a reusable username schema
            "type": "string", # Require a string value
            "description": "Original description", # Provide the original description
            "minLength": 3, # Require at least three characters
        }, # Finish defining the reusable schema
    }, # Finish defining the reusable components
    "properties": { # Define the root properties
        "username": { # Define the username property
            "$ref": "#/$defs/UserName", # Reference the reusable username schema
            "description": "Username used for login", # Override the referenced description
        }, # Finish defining the username property
    }, # Finish defining the properties
} # Finish creating the schema

resolved_override = dereference_refs(schema_with_override) # Resolve the reference and apply the override

resolved_override["properties"]["username"] # Display the final username schema

In [ ]:
# 3. Use a separate full_schema
full_schema = { # Define the complete schema containing reusable definitions
    "$defs": { # Define reusable schema components
        "Product": { # Define a reusable product schema
            "type": "object", # Specify that a product is an object
            "properties": { # Define product properties
                "name": {"type": "string"}, # Define the product name
                "price": {"type": "number"}, # Define the product price
            }, # Finish defining product properties
        }, # Finish defining the Product schema
    }, # Finish defining reusable components
} # Finish creating the complete schema

schema_fragment = { # Define a separate schema fragment
    "item": {"$ref": "#/$defs/Product"}, # Reference a definition from the complete schema
} # Finish creating the fragment

resolved_fragment = dereference_refs( # Resolve the fragment using the complete schema
    schema_fragment, # Provide the fragment to process
    full_schema=full_schema, # Provide the schema containing the referenced definition
) # Finish dereferencing the fragment

resolved_fragment # Display the resolved fragment

In [ ]:
# 4. Preserve selected keys without processing them
# By default, "$defs" is copied but not recursively dereferenced.

schema_with_definitions = { # Define a schema with nested references inside $defs
    "$defs": { # Define reusable components
        "Identifier": {"type": "integer"}, # Define an identifier schema
        "User": { # Define a user schema
            "properties": { # Define user properties
                "id": {"$ref": "#/$defs/Identifier"}, # Reference the identifier schema
            }, # Finish defining user properties
        }, # Finish defining the User schema
    }, # Finish defining reusable components
    "properties": { # Define root properties
        "user": {"$ref": "#/$defs/User"}, # Reference the User schema
    }, # Finish defining root properties
} # Finish creating the schema

default_result = dereference_refs(schema_with_definitions) # Resolve references while skipping recursive processing inside $defs

default_result["$defs"]["User"] # Show that the reference inside $defs remains unchanged

# To process every key, including "$defs", pass an empty sequence:
fully_resolved = dereference_refs( # Resolve references throughout the complete schema
    schema_with_definitions, # Provide the schema to process
    skip_keys=[], # Do not skip any keys
) # Finish dereferencing the schema

fully_resolved["$defs"]["User"] # Display the resolved definition


In [ ]:
# 5. Handle circular references safely
circular_schema = { # Define a schema containing a circular reference
    "$defs": { # Define reusable components
        "Node": { # Define a recursive node schema
            "type": "object", # Specify that a node is an object
            "properties": { # Define node properties
                "value": {"type": "string"}, # Define the node value
                "child": {"$ref": "#/$defs/Node"}, # Reference the same Node schema recursively
            }, # Finish defining node properties
        }, # Finish defining the Node schema
    }, # Finish defining reusable components
    "properties": { # Define root properties
        "root": {"$ref": "#/$defs/Node"}, # Reference the recursive Node schema
    }, # Finish defining root properties
} # Finish creating the circular schema

resolved_circular = dereference_refs(circular_schema) # Resolve references without entering infinite recursion

resolved_circular["properties"]["root"] # Display the safely resolved recursive schema